In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import os
from shutil import copyfile

In [3]:
TRIAL_DIR = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80'
DATA_DIR = os.path.join(TRIAL_DIR, 'data')

In [4]:
import os
if not os.path.exists(TRIAL_DIR):
  os.makedirs(TRIAL_DIR)
os.listdir(TRIAL_DIR)

if not os.path.exists(DATA_DIR):
  os.makedirs(DATA_DIR)
os.listdir(DATA_DIR)

['UCSF-PDGM-0432_nifti',
 'UCSF-PDGM-0433_FU007d_nifti',
 'UCSF-PDGM-0433_nifti',
 'UCSF-PDGM-0434_nifti',
 'UCSF-PDGM-0435_nifti',
 'UCSF-PDGM-0436_nifti',
 'UCSF-PDGM-0437_nifti',
 'UCSF-PDGM-0438_nifti',
 'UCSF-PDGM-0439_nifti',
 'UCSF-PDGM-0440_nifti',
 'UCSF-PDGM-0441_nifti',
 'UCSF-PDGM-0442_nifti',
 'UCSF-PDGM-0443_nifti',
 'UCSF-PDGM-0444_nifti',
 'UCSF-PDGM-0445_nifti',
 'UCSF-PDGM-0446_nifti',
 'UCSF-PDGM-0447_nifti',
 'UCSF-PDGM-0448_nifti',
 'UCSF-PDGM-0449_nifti',
 'UCSF-PDGM-0450_nifti',
 'UCSF-PDGM-0451_nifti',
 'UCSF-PDGM-0452_nifti',
 'UCSF-PDGM-0453_nifti',
 'UCSF-PDGM-0454_nifti',
 'UCSF-PDGM-0455_nifti',
 'UCSF-PDGM-0456_nifti',
 'UCSF-PDGM-0457_nifti',
 'UCSF-PDGM-0458_nifti',
 'UCSF-PDGM-0459_nifti',
 'UCSF-PDGM-0460_nifti',
 'UCSF-PDGM-0461_nifti',
 'UCSF-PDGM-0462_nifti',
 'UCSF-PDGM-0463_nifti',
 'UCSF-PDGM-0464_nifti',
 'UCSF-PDGM-0465_nifti',
 'UCSF-PDGM-0466_nifti',
 'UCSF-PDGM-0467_nifti',
 'UCSF-PDGM-0468_nifti',
 'UCSF-PDGM-0469_nifti',
 'UCSF-PDGM-0470_n

In [9]:
gbm_source_dir = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations'
lgg_source_dir = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations'
egd_source_dir = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/TEST1_EGD/xnat'
ucs_source_dir = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5'
rem_source_dir = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/raw_data/remind/'
utsw_source_dir = '/content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/'

In [11]:
"""
Inspect a sample of UTSW-Glioma cases to verify file structure,
geometry consistency, and segmentation label conventions BEFORE
processing the full 625-case cohort.

Outputs:
- Console summary of each case's files
- A CSV with per-case file inventory and geometry checks
- Key questions answered: Are r* and non-r* files in same space?
  What labels do segmentation files use? Are all cases consistent?
"""
!pip install SimpleITK
import os
import glob
import csv
import SimpleITK as sitk
import numpy as np

# =============================================================================
# 1. CONFIGURE
# =============================================================================
UTSW_ROOT = utsw_source_dir
N_CASES_TO_CHECK = 10  # start small, increase if needed
OUTPUT_CSV = 'utsw_inspection.csv'

# Get all case directories
all_case_dirs = sorted([d for d in glob.glob(os.path.join(UTSW_ROOT, 'BT*'))
                        if os.path.isdir(d)])
print(f"Total cases in UTSW-Glioma: {len(all_case_dirs)}")
print(f"Inspecting first {N_CASES_TO_CHECK} cases\n")


# =============================================================================
# 2. INSPECT EACH CASE
# =============================================================================
rows = []

for case_dir in all_case_dirs:
    case_id = os.path.basename(case_dir)
    print(f"========== {case_id} ==========")

    # List all .nii.gz files in this case folder
    all_niftis = sorted(glob.glob(os.path.join(case_dir, '*.nii.gz')))
    print(f"  Files present ({len(all_niftis)}):")
    for f in all_niftis:
        size_kb = os.path.getsize(f) / 1024
        print(f"    {os.path.basename(f):45s}  {size_kb:7.1f} KB")

    # Load the FeTS-stripped T1 as the reference geometry
    brain_t1_path = os.path.join(case_dir, 'brain_t1.nii.gz')
    if not os.path.exists(brain_t1_path):
        print(f"  WARNING: brain_t1.nii.gz missing, skipping geometry check")
        rows.append({'case_id': case_id, 'status': 'missing_brain_t1'})
        print()
        continue

    brain = sitk.ReadImage(brain_t1_path)
    ref_size = brain.GetSize()
    ref_spacing = tuple(round(s, 3) for s in brain.GetSpacing())
    ref_origin = tuple(round(o, 3) for o in brain.GetOrigin())

    print(f"\n  Reference (brain_t1.nii.gz):")
    print(f"    Size:    {ref_size}")
    print(f"    Spacing: {ref_spacing}")
    print(f"    Origin:  {ref_origin}")

    # Check geometry of every file relative to brain_t1
    print(f"\n  Geometry check (each file vs brain_t1):")
    row = {'case_id': case_id, 'status': 'ok'}

    files_to_check = [
        'brain_t1.nii.gz', 'brain_t1ce.nii.gz', 'brain_t2.nii.gz', 'brain_flair.nii.gz',
        'brain_t1_ants.nii.gz', 'brain_t1ce_ants.nii.gz', 'brain_t2_ants.nii.gz', 'brain_fl_ants.nii.gz',
        'tumorseg_FeTS.nii.gz',
        'tumorseg_manual_correction.nii.gz',
        'rtumorseg_manual_correction.nii.gz',
    ]

    for fname in files_to_check:
        fpath = os.path.join(case_dir, fname)
        if not os.path.exists(fpath):
            row[fname] = 'absent'
            continue

        img = sitk.ReadImage(fpath)
        size = img.GetSize()
        spacing = tuple(round(s, 3) for s in img.GetSpacing())

        size_match = (size == ref_size)
        spacing_match = (spacing == ref_spacing)
        geometry_ok = size_match and spacing_match

        status = 'MATCH' if geometry_ok else 'MISMATCH'
        size_str = f"size={size}" if not size_match else ""
        spacing_str = f"spacing={spacing}" if not spacing_match else ""
        print(f"    {fname:45s} {status} {size_str} {spacing_str}")
        row[fname] = status

        # For segmentation files, also report unique label values and voxel counts
        if fname.startswith(('tumorseg', 'rtumorseg')):
            arr = sitk.GetArrayFromImage(img)
            unique_vals = np.unique(arr).tolist()
            wt_voxels = int(np.sum(arr > 0))
            print(f"      {'':47s}labels={unique_vals}, WT voxels={wt_voxels}")
            row[f'{fname}_labels'] = str(unique_vals)
            row[f'{fname}_wt_voxels'] = wt_voxels

    rows.append(row)
    print()


# =============================================================================
# 3. WRITE SUMMARY CSV
# =============================================================================
if rows:
    all_keys = sorted(set().union(*[r.keys() for r in rows]))
    with open(OUTPUT_CSV, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_keys)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print(f"Inspection CSV written: {OUTPUT_CSV}")

Total cases in UTSW-Glioma: 625
Inspecting first 10 cases

========== BT0001 ==========
  Files present (11):
    brain_fl_ants.nii.gz                            4194.9 KB
    brain_flair.nii.gz                              4752.8 KB
    brain_t1.nii.gz                                 4669.6 KB
    brain_t1_ants.nii.gz                            4110.8 KB
    brain_t1ce.nii.gz                               4673.9 KB
    brain_t1ce_ants.nii.gz                          4115.5 KB
    brain_t2.nii.gz                                 4839.3 KB
    brain_t2_ants.nii.gz                            4273.7 KB
    rtumorseg_manual_correction.nii.gz                32.1 KB
    tumorseg_FeTS.nii.gz                              33.3 KB
    tumorseg_manual_correction.nii.gz                 17.7 KB

  Reference (brain_t1.nii.gz):
    Size:    (240, 240, 155)
    Spacing: (1.0, 1.0, 1.0)
    Origin:  (0.0, -239.0, 0.0)

  Geometry check (each file vs brain_t1):
    brain_t1.nii.gz                        

KeyboardInterrupt: 

In [ ]:
# First let's take tcga gbm


def run_gbm_data_copy():

    for patient in os.listdir(gbm_source_dir):
        print(patient)
        if patient == '.ipynb_checkpoints':
            continue
        print(patient)
        flair_path = ''
        t1_path = ''
        t1ce_path = ''
        t2_path = ''
        seg_path = ''

        #print(os.listdir(os.path.join(gbm_source_dir, patient)))
        for series in os.listdir(os.path.join(gbm_source_dir, patient)):
            if 'flair.nii.gz' in series:
                flair_path = os.path.join(gbm_source_dir, patient, series)
            if 't1.nii.gz' in series:
                t1_path = os.path.join(gbm_source_dir, patient, series)
            if 't1Gd.nii.gz' in series:
                t1ce_path = os.path.join(gbm_source_dir, patient, series)
            if 't2.nii.gz' in series:
                t2_path = os.path.join(gbm_source_dir, patient, series)
            if 'GlistrBoost.nii.gz' in series:
                seg_path = os.path.join(gbm_source_dir, patient, series)

        for series in os.listdir(os.path.join(gbm_source_dir, patient)):
            if 'GlistrBoost_ManuallyCorrected.nii.gz' in series:
                seg_path = os.path.join(gbm_source_dir, patient, series)
        print("flair_path: ", flair_path)
        print("t1_path: ", t1_path)
        print("t1ce_path: ", t1ce_path)
        print("t2_path: ", t2_path)
        print("seg_path: ", seg_path)



        os.makedirs(os.path.join(DATA_DIR, patient), exist_ok=True)
        copyfile(flair_path, os.path.join(DATA_DIR, patient, 'flair.nii.gz'))
        copyfile(t1_path, os.path.join(DATA_DIR, patient, 't1.nii.gz'))
        copyfile(t1ce_path, os.path.join(DATA_DIR, patient, 't1ce.nii.gz'))
        copyfile(t2_path, os.path.join(DATA_DIR, patient, 't2.nii.gz'))
        copyfile(seg_path, os.path.join(DATA_DIR, patient, 'seg.nii.gz'))

run_gbm_data_copy()

TCGA-76-6657
TCGA-76-6657
flair_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/TCGA-76-6657/TCGA-76-6657_2001.06.11_flair.nii.gz
t1_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/TCGA-76-6657/TCGA-76-6657_2001.06.11_t1.nii.gz
t1ce_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/TCGA-76-6657/TCGA-76-6657_2001.06.11_t1Gd.nii.gz
t2_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/Pre-operative_TCGA_GBM_NIfTI_and_Segmentations/TCG

In [ ]:
# Now let's run LGG

from shutil import copyfile

def run_lgg_data_copy():

    for patient in os.listdir(lgg_source_dir):
        if patient == '.ipynb_checkpoints':
            continue
        print(patient)
        flair_path = ''
        t1_path = ''
        t1ce_path = ''
        t2_path = ''
        seg_path = ''

        for series in os.listdir(os.path.join(lgg_source_dir, patient)):
            if 'flair.nii.gz' in series:
                flair_path = os.path.join(lgg_source_dir, patient, series)
            if 't1.nii.gz' in series:
                t1_path = os.path.join(lgg_source_dir, patient, series)
            if 't1Gd.nii.gz' in series:
                t1ce_path = os.path.join(lgg_source_dir, patient, series)
            if 't2.nii.gz' in series:
                t2_path = os.path.join(lgg_source_dir, patient, series)
            if 'GlistrBoost.nii.gz' in series:
                seg_path = os.path.join(lgg_source_dir, patient, series)

        for series in os.listdir(os.path.join(lgg_source_dir, patient)):
            if 'GlistrBoost_ManuallyCorrected.nii.gz' in series:
                seg_path = os.path.join(lgg_source_dir, patient, series)
        print("flair_path: ", flair_path)
        print("t1_path: ", t1_path)
        print("t1ce_path: ", t1ce_path)
        print("t2_path: ", t2_path)
        print("seg_path: ", seg_path)



        os.makedirs(os.path.join(DATA_DIR, patient), exist_ok=True)
        copyfile(flair_path, os.path.join(DATA_DIR, patient, 'flair.nii.gz'))
        copyfile(t1_path, os.path.join(DATA_DIR, patient, 't1.nii.gz'))
        copyfile(t1ce_path, os.path.join(DATA_DIR, patient, 't1ce.nii.gz'))
        copyfile(t2_path, os.path.join(DATA_DIR, patient, 't2.nii.gz'))
        copyfile(seg_path, os.path.join(DATA_DIR, patient, 'seg.nii.gz'))

run_lgg_data_copy()

TCGA-HT-A61A
flair_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/TCGA-HT-A61A/TCGA-HT-A61A_2000.01.27_flair.nii.gz
t1_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/TCGA-HT-A61A/TCGA-HT-A61A_2000.01.27_t1.nii.gz
t1ce_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/TCGA-HT-A61A/TCGA-HT-A61A_2000.01.27_t1Gd.nii.gz
t2_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV1_TCGA_GBM_AND_LGG/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/Pre-operative_TCGA_LGG_NIfTI_and_Segmentations/TCGA-HT-A61A/TCG

In [ ]:
# Now let's run UCSF

from shutil import copyfile

def run_ucs_data_copy():

    for patient in os.listdir(ucs_source_dir):
        print(patient)
        if patient == '.ipynb_checkpoints':
            continue
        flair_path = ''
        t1_path = ''
        t1ce_path = ''
        t2_path = ''
        seg_path = ''
        for series in os.listdir(os.path.join(ucs_source_dir, patient)):
            if 'FLAIR.nii.gz' in series:
                flair_path = os.path.join(ucs_source_dir, patient, series)
            if 'T1.nii.gz' in series:
                t1_path = os.path.join(ucs_source_dir, patient, series)
            if 'T1c.nii.gz' in series:
                t1ce_path = os.path.join(ucs_source_dir, patient, series)
            if 'T2.nii.gz' in series:
                t2_path = os.path.join(ucs_source_dir, patient, series)
            if 'tumor_segmentation.nii.gz' in series:
                seg_path = os.path.join(ucs_source_dir, patient, series)

                seg_path = os.path.join(ucs_source_dir, patient, series)
        print("flair_path: ", flair_path)
        print("t1_path: ", t1_path)
        print("t1ce_path: ", t1ce_path)
        print("t2_path: ", t2_path)
        print("seg_path: ", seg_path)



        os.makedirs(os.path.join(DATA_DIR, patient), exist_ok=True)
        copyfile(flair_path, os.path.join(DATA_DIR, patient, 'flair.nii.gz'))
        copyfile(t1_path, os.path.join(DATA_DIR, patient, 't1.nii.gz'))
        copyfile(t1ce_path, os.path.join(DATA_DIR, patient, 't1ce.nii.gz'))
        copyfile(t2_path, os.path.join(DATA_DIR, patient, 't2.nii.gz'))
        copyfile(seg_path, os.path.join(DATA_DIR, patient, 'seg.nii.gz'))

run_ucs_data_copy()

UCSF-PDGM-0004_nifti
flair_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_FLAIR.nii.gz
t1_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_T1.nii.gz
t1ce_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_T1c.nii.gz
t2_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_T2.nii.gz
seg_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-v5/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_tumor_segmentation.nii.gz
UCSF-PDGM-0005_nifti
flair_path:  /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_50/raw_data/DEV2_UCSF-PDGM/UCSF-PDGM-

In [ ]:
# EGD

from shutil import copyfile

egd_expert_seg_cases = pd.read_excel('/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/cohorts/egd/resources/PROJECT_DATA/Segmentation_source.xlsx')

# filtered_egd_cases = egd_expert_seg_cases[egd_expert_seg_cases['Observer'] != 'AUTO']
# egd_subject_ids = filtered_egd_cases['Subject'].tolist()

egd_subject_ids = egd_expert_seg_cases['Subject'].tolist()
egd_subject_ids = [str("MR_")+x for x in egd_subject_ids]
print(egd_subject_ids)

print(len(egd_subject_ids))


# Now let's run LGG

from shutil import copyfile

def run_egd_data_copy():

    for patient in os.listdir(egd_source_dir):
        if patient == '.ipynb_checkpoints':
            continue
        if os.path.exists(os.path.join(DATA_DIR, patient)):
            continue
        print(patient)
        flair_path = ''
        t1_path = ''
        t1ce_path = ''
        t2_path = ''
        seg_path = ''

        if patient not in egd_subject_ids:
            continue


        flair_path = os.path.join(egd_source_dir, patient, "scans", "4_FLAIR-FLAIR", "resources", "NIFTI", "files", "FLAIR.nii.gz")
        t1_path = os.path.join(egd_source_dir, patient, "scans", "1_T1-T1", "resources", "NIFTI", "files", "T1.nii.gz")
        t1ce_path = os.path.join(egd_source_dir, patient, "scans", "2_T1GD-T1GD", "resources", "NIFTI", "files", "T1GD.nii.gz")
        t2_path = os.path.join(egd_source_dir, patient, "scans", "3_T2-T2", "resources", "NIFTI", "files", "T2.nii.gz")
        seg_path = os.path.join(egd_source_dir, patient, "scans", "5_MASK-MASK", "resources", "NIFTI", "files", "MASK.nii.gz")

        print("flair_path: ", flair_path)
        print("t1_path: ", t1_path)
        print("t1ce_path: ", t1ce_path)
        print("t2_path: ", t2_path)
        print("seg_path: ", seg_path)



        os.makedirs(os.path.join(DATA_DIR, patient), exist_ok=True)
        copyfile(flair_path, os.path.join(DATA_DIR, patient, 'flair.nii.gz'))
        copyfile(t1_path, os.path.join(DATA_DIR, patient, 't1.nii.gz'))
        copyfile(t1ce_path, os.path.join(DATA_DIR, patient, 't1ce.nii.gz'))
        copyfile(t2_path, os.path.join(DATA_DIR, patient, 't2.nii.gz'))
        copyfile(seg_path, os.path.join(DATA_DIR, patient, 'seg.nii.gz'))

run_egd_data_copy()

['MR_EGD-0001', 'MR_EGD-0002', 'MR_EGD-0003', 'MR_EGD-0004', 'MR_EGD-0005', 'MR_EGD-0006', 'MR_EGD-0007', 'MR_EGD-0008', 'MR_EGD-0009', 'MR_EGD-0010', 'MR_EGD-0011', 'MR_EGD-0012', 'MR_EGD-0013', 'MR_EGD-0014', 'MR_EGD-0015', 'MR_EGD-0016', 'MR_EGD-0017', 'MR_EGD-0018', 'MR_EGD-0019', 'MR_EGD-0020', 'MR_EGD-0021', 'MR_EGD-0022', 'MR_EGD-0023', 'MR_EGD-0024', 'MR_EGD-0025', 'MR_EGD-0026', 'MR_EGD-0027', 'MR_EGD-0028', 'MR_EGD-0029', 'MR_EGD-0030', 'MR_EGD-0031', 'MR_EGD-0032', 'MR_EGD-0033', 'MR_EGD-0034', 'MR_EGD-0035', 'MR_EGD-0036', 'MR_EGD-0037', 'MR_EGD-0038', 'MR_EGD-0039', 'MR_EGD-0040', 'MR_EGD-0041', 'MR_EGD-0042', 'MR_EGD-0043', 'MR_EGD-0044', 'MR_EGD-0045', 'MR_EGD-0046', 'MR_EGD-0047', 'MR_EGD-0048', 'MR_EGD-0049', 'MR_EGD-0050', 'MR_EGD-0051', 'MR_EGD-0052', 'MR_EGD-0053', 'MR_EGD-0054', 'MR_EGD-0055', 'MR_EGD-0056', 'MR_EGD-0057', 'MR_EGD-0058', 'MR_EGD-0059', 'MR_EGD-0060', 'MR_EGD-0061', 'MR_EGD-0062', 'MR_EGD-0063', 'MR_EGD-0064', 'MR_EGD-0065', 'MR_EGD-0066', 'MR_EGD-0

In [16]:
# First let's take tcga gbm


def run_utsw_data_copy():

    for patient in os.listdir(utsw_source_dir):
        print(patient)
        if patient == '.ipynb_checkpoints':
            continue
        print(patient)
        flair_path = ''
        t1_path = ''
        t1ce_path = ''
        t2_path = ''
        seg_path = ''

        #print(os.listdir(os.path.join(utsw_source_dir, patient)))
        for series in os.listdir(os.path.join(utsw_source_dir, patient)):
            if 'brain_flair.nii.gz' in series:
                flair_path = os.path.join(utsw_source_dir, patient, series)
            if 'brain_t1.nii.gz' in series:
                t1_path = os.path.join(utsw_source_dir, patient, series)
            if 'brain_t1ce.nii.gz' in series:
                t1ce_path = os.path.join(utsw_source_dir, patient, series)
            if 'brain_t2.nii.gz' in series:
                t2_path = os.path.join(utsw_source_dir, patient, series)
            if 'tumorseg_FeTS.nii.gz' in series:
                seg_path = os.path.join(utsw_source_dir, patient, series)

        for series in os.listdir(os.path.join(utsw_source_dir, patient)):
            if 'rtumorseg_manual_correction.nii.gz' in series:
                seg_path = os.path.join(utsw_source_dir, patient, series)
        print("flair_path: ", flair_path)
        print("t1_path: ", t1_path)
        print("t1ce_path: ", t1ce_path)
        print("t2_path: ", t2_path)
        print("seg_path: ", seg_path)



        os.makedirs(os.path.join(DATA_DIR, patient), exist_ok=True)
        copyfile(flair_path, os.path.join(DATA_DIR, patient, 'flair.nii.gz'))
        copyfile(t1_path, os.path.join(DATA_DIR, patient, 't1.nii.gz'))
        copyfile(t1ce_path, os.path.join(DATA_DIR, patient, 't1ce.nii.gz'))
        copyfile(t2_path, os.path.join(DATA_DIR, patient, 't2.nii.gz'))
        copyfile(seg_path, os.path.join(DATA_DIR, patient, 'seg.nii.gz'))

run_utsw_data_copy()

BT0001
BT0001
flair_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0001/brain_flair.nii.gz
t1_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0001/brain_t1.nii.gz
t1ce_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0001/brain_t1ce.nii.gz
t2_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0001/brain_t2.nii.gz
seg_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0001/rtumorseg_manual_correction.nii.gz
BT0002
BT0002
flair_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0002/brain_flair.nii.gz
t1_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT0002/brain_t1.nii.gz
t1ce_path:  /content/drive/MyDrive/bsd/00_tcia_datasets/UTSW-GLIOMA/PKG - UTSW-Glioma/UTSW-Glioma/BT000